# Análisis de errores: TartuNLP-ru → dev SV (Subtask 2)

Solo CPU. Necesita el fichero de predicciones `SV.tartunlp_ru.tsv` (salida de `predict_tartunlp.py`) guardado en Drive.

0. **Métrica oficial (Average F1)**: dónde se pierde el F1
1. AUC por palabra y periodo
2. AUC por tipo de hipótesis (denotativa / social)
3. Errores más seguros (FP/FN) → TSV para clasificar a mano
4. ¿Se guía el modelo por palabras compartidas entre definición y frase?

In [2]:
from google.colab import drive
drive.mount('/content/drive')
PRED = '/content/drive/MyDrive/semeval27/preds/SV.tartunlp_ru.tsv'
OUT_DIR = '/content/drive/MyDrive/semeval27/results'

import os, re
import pandas as pd
from collections import Counter
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option('display.max_colwidth', 250)

df = pd.read_csv(PRED, sep='\t', dtype={'sentence_id': str})
df['label'] = df['label'].astype(int)

def auc(g):
    return roc_auc_score(g['label'], g['prob']) if g['label'].nunique() == 2 else float('nan')

print(len(df), 'usos,', df['word'].nunique(), 'palabras')

Mounted at /content/drive
1983 usos, 10 palabras


## 0. Métrica oficial: Average F1
La tarea se evalúa con **"Average F1"** de la clase positiva (¿expresa el uso el sentido definido?). Todavía no está confirmado si se promedia **por palabra** o **globalmente**, así que calculamos las dos lecturas (igual que `evaluate_subtask2.py`).

Además de comparar con la baseline "siempre 1", miramos:
- **FP / FN con umbral 0,35:** qué tipo de error cuesta más F1 en cada palabra.
- **F1 con el mejor umbral por palabra (oráculo):** el máximo que se podría sacar con la *ordenación* actual del modelo si acertáramos el umbral de cada palabra. Es optimista (umbral elegido con las etiquetas), pero dice si merece la pena trabajar la calibración o hay que mejorar el modelo.

In [3]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
UMBRAL = 0.35
df['pred'] = (df['prob'] >= UMBRAL).astype(int)

def f1(y, p): return f1_score(y, p, zero_division=0)

def mejor_f1(g):
    return max(f1(g['label'], (g['prob'] >= t).astype(int)) for t in np.unique(g['prob']))

rep = df.groupby('word').apply(lambda g: pd.Series({
    'n': len(g), 'pos_rate': g['label'].mean(), 'pred_rate': g['pred'].mean(),
    'FP': int(((g['pred'] == 1) & (g['label'] == 0)).sum()),
    'FN': int(((g['pred'] == 0) & (g['label'] == 1)).sum()),
    'P': precision_score(g['label'], g['pred'], zero_division=0),
    'R': recall_score(g['label'], g['pred'], zero_division=0),
    'F1@0.35': f1(g['label'], g['pred']),
    'F1 siempre1': f1(g['label'], np.ones(len(g), int)),
    'F1 oráculo': mejor_f1(g),
})).round(3)
print(rep.to_string())

tabla = pd.DataFrame({
    'F1 macro (por palabra)': [rep['F1@0.35'].mean(), rep['F1 siempre1'].mean(), rep['F1 oráculo'].mean()],
    'F1 global': [f1(df['label'], df['pred']), f1(df['label'], np.ones(len(df), int)), np.nan],
}, index=['TartuNLP-ru @0.35', 'Baseline siempre 1', 'Oráculo umbral por palabra']).round(3)
print('\n', tabla.to_string())

               n  pos_rate  pred_rate     FP    FN      P      R  F1@0.35  F1 siempre1  F1 oráculo
word                                                                                              
dumpa      145.0     0.310      0.469   51.0  28.0  0.250  0.378    0.301        0.474       0.476
exportera  149.0     0.946      0.946    7.0   7.0  0.950  0.950    0.950        0.972       0.972
fru        274.0     0.894      0.748   25.0  65.0  0.878  0.735    0.800        0.944       0.944
fröken     174.0     0.057      0.868  142.0   1.0  0.060  0.900    0.112        0.109       0.159
förort     150.0     0.493      0.787   53.0   9.0  0.551  0.878    0.677        0.661       0.693
herre      283.0     0.071      1.000  263.0   0.0  0.071  1.000    0.132        0.132       0.413
kvinna     290.0     0.228      0.862  185.0   1.0  0.260  0.985    0.411        0.371       0.492
motiv      291.0     0.182      0.986  235.0   1.0  0.181  0.981    0.306        0.308       0.383
skär      

/tmp/ipykernel_3370/3062693749.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rep = df.groupby('word').apply(lambda g: pd.Series({


## 1. AUC por palabra y periodo
¿El modelo funciona peor en los textos antiguos (OCR, ortografía)? Si un periodo tiene casi todo 0 o casi todo 1, sale NaN: no se puede medir.

In [4]:
t = (df.groupby(['word', 'period_label'])
       .apply(lambda g: pd.Series({'n': len(g), 'positivos': g['label'].mean(), 'auc': auc(g)}))
       .round(3))
print(t.to_string())
print('\nAUC medio por periodo (sobre palabras):')
print(t.groupby('period_label')['auc'].mean().round(3))

                           n  positivos    auc
word      period_label                        
dumpa     2006-2007     30.0      0.467  0.290
          2010-2011     28.0      0.429  0.391
          2014-2015     27.0      0.222  0.683
          2018-2019     30.0      0.167  0.608
          2022-2023     30.0      0.267  0.625
exportera 2006-2007     30.0      0.900  0.593
          2010-2011     30.0      1.000    NaN
          2014-2015     29.0      0.931  0.500
          2018-2019     30.0      1.000    NaN
          2022-2023     30.0      0.900  0.494
fru       1880-1883     29.0      0.724  0.446
          1888-1891     26.0      0.885  0.217
          1896-1899     23.0      0.826  0.224
          1904-1907     18.0      0.667  0.375
          1912-1915     28.0      0.821  0.313
          2006-2007     30.0      1.000    NaN
          2010-2011     30.0      0.967  0.414
          2014-2015     30.0      1.000    NaN
          2018-2019     30.0      0.967  0.379
          202

/tmp/ipykernel_3370/4212321275.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({'n': len(g), 'positivos': g['label'].mean(), 'auc': auc(g)}))


## 2. AUC por tipo de hipótesis
**Revisad la clasificación leyendo cada definición** (se imprimen abajo) y anotad el criterio. Con 5 palabras por grupo no se puede afirmar una diferencia general, solo si los datos contradicen la idea de que "lo social es más difícil".

In [5]:
print(df.groupby('word')['definition'].first().to_string())

word
dumpa                                                                                                                                                                                      Låta något falla; lägga eller sänka ner
exportera                                                                                                                                                                     Föra ut en produkt ur ett land (till ett annat land)
fru                                                                                                                                                                                                   Maka, hustru (oavsett klass)
fröken                                                                                                                                   Gift eller ogift yrkesutövande kvinna, företrädesvis i skola, butik, postkontor, och bank
förort                                                                                 

In [6]:
tipo = {   # <- REVISAD
    'motiv': 'denotativa', 'suga': 'denotativa', 'dumpa': 'denotativa',
    'exportera': 'denotativa', 'skär': 'denotativa',
    'herre': 'social', 'fru': 'social', 'fröken': 'social',
    'kvinna': 'social', 'förort': 'social',
}
res = df.groupby('word').apply(lambda g: pd.Series({
    'n': len(g), 'pos_rate': g['label'].mean(), 'auc': auc(g)})).round(3)
res['tipo'] = res.index.map(tipo)
print(res.sort_values(['tipo', 'auc']).to_string())
print('\nAUC medio por tipo:')
print(res.groupby('tipo')['auc'].agg(['mean', 'count']).round(3))

               n  pos_rate    auc        tipo
word                                         
dumpa      145.0     0.310  0.465  denotativa
exportera  149.0     0.946  0.492  denotativa
skär        98.0     0.276  0.524  denotativa
suga       129.0     0.186  0.585  denotativa
motiv      291.0     0.182  0.612  denotativa
fru        274.0     0.894  0.381      social
fröken     174.0     0.057  0.562      social
förort     150.0     0.493  0.635      social
kvinna     290.0     0.228  0.726      social
herre      283.0     0.071  0.847      social

AUC medio por tipo:
             mean  count
tipo                    
denotativa  0.536      5
social      0.630      5


/tmp/ipykernel_3370/1056314426.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = df.groupby('word').apply(lambda g: pd.Series({


## 3. Errores más seguros → clasificación a mano
- **FP:** probabilidad alta pero etiqueta 0. **FN:** probabilidad baja pero etiqueta 1.
- Ojo: con umbral 0,35 casi todo es positivo, así que los FP son los errores que de verdad cuestan F1 (columna `pred`). Los FN más seguros sirven para ver qué no entiende el modelo.
- Se guardan en Drive (`error_analysis_SV.tsv`). Abridlo en Google Sheets y poned una categoría a cada fila:

| Cat. | Significa |
|---|---|
| A | Otro sentido claro que el modelo no distingue |
| B | OCR / ortografía antigua |
| C | Pista superficial (palabras compartidas con la definición) |
| D | Ambiguo incluso para una persona |
| E | Posible error de anotación (argumentarlo) |
| F | Falta contexto |

Para hoy basta con 2 palabras (p. ej. *motiv* y *herre*).

In [7]:
def errores(palabra, k=10):
    g = df[df['word'] == palabra]
    fp = g[g['label'] == 0].nlargest(k, 'prob')
    fn = g[g['label'] == 1].nsmallest(k, 'prob')
    return pd.concat([fp.assign(error='FP'), fn.assign(error='FN')])

cols = ['word', 'error', 'period_label', 'prob', 'label', 'pred', 'target', 'text']  # pred = decisión con umbral 0,35
err = pd.concat([errores(w) for w in sorted(df['word'].unique())])[cols]
err['categoria'] = ''
err['comentario'] = ''
err.to_csv(f'{OUT_DIR}/error_analysis_SV.tsv', sep='\t', index=False)
print('Guardado:', f'{OUT_DIR}/error_analysis_SV.tsv', '-', len(err), 'filas')

print('\nDefinición de motiv:', df.loc[df['word'] == 'motiv', 'definition'].iloc[0])
errores('motiv', 5)[['error', 'prob', 'period_label', 'text']]

Guardado: /content/drive/MyDrive/semeval27/results/error_analysis_SV.tsv - 198 filas

Definición de motiv: Ämne för konstnärlig fram­ställning in­om bild­konst och skönlitteratur


,error,prob,period_label,text
1974,FP,0.807098,1880-1883,"Från dessa trakter är det , man hör talas om öfverfall och våldsgerningar , hvilkas enda motiv emellanåt synes vara den mest djuriska råhet ."
1864,FP,0.806324,1912-1915,"Väggarna äro som för öfrigt allt på denna utställning dekorerade i behagligt enkel stilsmak , här med ett motiv af starkt stiliserade uddiga kronor i guld och färger ."
1826,FP,0.802803,1912-1915,"Genom utrikesdepartementet har öfversttthallareämbetet i dagarna fått nig tillsändt af skrift af ett aktstycke , innehållande motiven till schweiziska » Bundesgerichts> afkunnande utslag i utlämningsfrågan ."
1827,FP,0.801789,1880-1883,"Häri är inflätadt ett af de ledande motiven , som redan förekommer i introduktionen ( ouverturen ) och som kan betraktas som Graalmotivet ."
1710,FP,0.801439,1888-1891,"Inför en dylik förtegen välgörenhet böjer menniskovännen med vördnad sitt hufvud , viss att inga fafängans motiv grumla den hjertekälla , hvarur den goda gerningen qväller ."
1741,FN,0.337771,1888-1891,Schultzberg » Mot aftonen » S Nilsson » Månat staten bör så snart sken » C F Svensson » Motiv från Holland- * en » °skilde« kraf auagalagte genast ersätta .TamvagSOlyCka 1 Förenta Staterna ( honom och sedan i sin ordning halla sig till
1696,FN,0.531637,1880-1883,"A . Gödecke , Vexiö : Motiv från Dufnäs af A . Schultzberg ; frii J . Gidlöf , Sundsvall : Rik skörd af Jenuy Nyström ; handlanden O. Petersson , Jönköping : Lunta af Christin Sundberg ; postmästaren C ."
1758,FN,0.571713,1912-1915,"gottlänska landskap , kyrkoruiner i Wisby och en hel serie bilder från dess botaniska trädgård , äfvensom ett flertel blad från fastlandet och bland dessa flera vackra motiv från Karlskrona ."
1930,FN,0.606808,1880-1883,"ett motsvarande mytiskt-religiöst motiv , och sålunda har gndarnes sirid mot giganterna , lapiternas mot kentaurerna , The sus ' mot amasonerna fleifaldiga gånger arn vändts för att förhärliga den grekiska civilisationens segrar öfver Orientens b..."
1949,FN,0.608464,1888-1891,"Wallander ; 2:o » Elgskytte för bandhund » , oljemålning af Johan Tirén ; samt 3:o ) » Höstdag efter dimma » , motiv från Södermanland , oljemålning af Edvard Bosenberg ."


Cuando hayáis rellenado la columna `categoria` en Sheets, descargad el TSV, sustituid el de Drive y ejecutad esto para el recuento:

In [17]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
!find /content/drive/MyDrive -name "error_analysis*"

Mounted at /content/drive
/content/drive/MyDrive/semeval27/results/error_analysis_SV_pre.tsv
/content/drive/MyDrive/semeval27/results/error_analysis_SV (1).tsv


In [20]:
done = pd.read_csv(f'{OUT_DIR}/error_analysis_SV (1).tsv', sep='\t').dropna(subset=['categoria'])
if len(done):
    print(pd.crosstab([done['word'], done['error']], done['categoria'], margins=True))
else:
    print('Todavía no hay categorías rellenadas.')

categoria         A  B   D   E  All
word      error                    
dumpa     FN     10  0   0   0   10
          FP      8  0   0   2   10
exportera FN      7  0   0   0    7
          FP      6  0   1   0    7
fru       FN      4  1   1   4   10
          FP      2  0   1   7   10
fröken    FN      1  0   0   0    1
          FP      6  0   0   4   10
förort    FN      9  0   0   0    9
          FP      6  0   4   0   10
herre     FP      2  0   4   4   10
kvinna    FN      0  1   0   0    1
          FP      9  0   0   1   10
motiv     FN      0  1   0   0    1
          FP      5  0   4   1   10
skär      FN      2  0   1   0    3
          FP      9  0   0   1   10
suga      FP     10  0   0   0   10
All              96  3  16  24  139


## 4. ¿Entiende el sentido o usa pistas superficiales? (*human-centered*)
**4a.** `solape` = fracción de palabras de la definición (4+ letras) que aparecen en la frase.
Si `rho(prob, solape)` es claramente mayor que `rho(label, solape)`, el modelo sigue el solapamiento más de lo que este predice la respuesta correcta: indicio de atajo.

In [21]:
def toks(s):
    return set(re.findall(r'[a-zåäö]{4,}', str(s).lower()))

df['solape'] = [len(toks(d) & toks(t)) / max(len(toks(d)), 1)
                for d, t in zip(df['definition'], df['text'])]
rows = []
for w, g in df.groupby('word'):
    rows.append({'word': w,
                 'rho(prob, solape)': spearmanr(g['prob'], g['solape']).statistic,
                 'rho(label, solape)': spearmanr(g['label'], g['solape']).statistic,
                 'auc': auc(g)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

     word  rho(prob, solape)  rho(label, solape)   auc
    dumpa             -0.001               0.112 0.465
exportera              0.117              -0.036 0.492
      fru              0.133              -0.078 0.381
   fröken             -0.188              -0.051 0.562
   förort              0.029              -0.109 0.635
    herre              0.042               0.055 0.847
   kvinna              0.262               0.054 0.726
    motiv              0.018               0.051 0.612
     skär              0.323               0.094 0.524
     suga              0.133               0.184 0.585


**4b.** ¿Qué palabras acompañan a los usos que el modelo puntúa más alto (20 % superior) frente a los más bajos (20 % inferior)? Probad con *herre* (donde mejor va) y luego *förort*, *fröken*, *kvinna*. Si salen palabras de contexto social y no del sentido, el modelo asocia más que entiende.

In [22]:
def palabras_asociadas(palabra, n=20):
    g = df[df['word'] == palabra].copy()
    g['tokens'] = g['text'].str.lower().apply(lambda s: set(re.findall(r'[a-zåäö]{3,}', s)))
    alto = g[g['prob'] >= g['prob'].quantile(.8)]
    bajo = g[g['prob'] <= g['prob'].quantile(.2)]
    ca = Counter(t for s in alto['tokens'] for t in s)
    cb = Counter(t for s in bajo['tokens'] for t in s)
    dif = {t: ca[t] / len(alto) - cb[t] / len(bajo) for t in set(ca) | set(cb)}
    print(f'[{palabra}] típicas de prob ALTA:', sorted(dif, key=dif.get, reverse=True)[:n])
    print(f'[{palabra}] típicas de prob BAJA:', sorted(dif, key=dif.get)[:n], '\n')

for w in ['herre', 'förort', 'fröken', 'kvinna']:
    palabras_asociadas(w)

[herre] típicas de prob ALTA: ['som', 'han', 'herre', 'sig', 'den', 'ett', 'till', 'men', 'honom', 'var', 'hade', 'damer', 'annan', 'hvilka', 'genom', 'hur', 'hvilken', 'och', 'att', 'icke']
[herre] típicas de prob BAJA: ['herrarnas', 'kommer', 'samt', 'klockan', 'ska', 'eller', 'säger', 'efter', 'sirius', 'hos', 'svenska', 'bästa', 'söndagen', 'söndag', 'skriver', 'meter', 'herr', 'stort', 'klubben', 'våren'] 

[förort] típicas de prob ALTA: ['det', 'har', 'förorterna', 'men', 'inte', 'finns', 'ett', 'att', 'innerstad', 'landsbygden', 'hur', 'från', 'ska', 'stockholm', 'mötesplatser', 'sker', 'små', 'samhället', 'stad', 'några']
[förort] típicas de prob BAJA: ['den', 'till', 'enligt', 'damaskus', 'sig', 'två', 'förorter', 'norra', 'senaste', 'förort', 'stort', 'sköt', 'paris', 'utan', 'män', 'livet', 'flera', 'antal', 'ledare', 'sarin'] 

[fröken] típicas de prob ALTA: ['fru', 'stockholm', 'anna', 'kalmar', 'malmö', 'von', 'hon', 'fröknarna', 'andersson', 'doktorinnan', 'sista', 'hilm